# NSOS — Suíte de generalização (padrão-ouro) v4: recall + extrapolação

Fase 1 da avaliação de **arquitetura** (branch `nsos-gpu-phases12`).

**Achados das runs v1–v3 (incorporados aqui):**
- O **harness funciona** — a tarefa **selective copy aprendeu** (mamba 0.99, híbrido
  0.78, atenção 0.69 no comprimento de treino) e o overfit de 1 exemplo vai a loss 0.
- **Não há gap ternário na GPU:** float == ternário idênticos (na GPU o forward é
  float/reference path em treino E inferência; o 1.58-bit só morde no caminho
  empacotado CPU/dp4a). Eval agora é único.
- **Primeiro sinal de arquitetura (copy):** o **Mamba extrapola melhor** em
  comprimento (L=128: mamba 0.33 vs atenção 0.18) — a tese do SSM.
- O **AR (recall) não treinava** porque minha formulação tinha chave/valor no MESMO
  vocab (o modelo tinha que inferir chave-vs-valor por posição) + pouco sinal/steps.

**v4:** AR com **vocabulários DISJUNTOS** (chave vs valor por faixa de id) + **mais
steps e LR maior** (recall é mais difícil que copy); eval único; gate = atenção
resolve AR n_kv=8. Tarefas 100% sintéticas (sem tokenizer/download). Compara o mixer:
atenção-pura / Mamba-puro / híbrido NSOS.

## 1 — GPU + (opcional) Drive p/ cache do build

In [ ]:
import os, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/nsos_genbench'); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except Exception as e:
    print('[drive] indisponivel (ok):', e); DRIVE_ROOT = Path('/content/nsos_genbench'); DRIVE_ROOT.mkdir(exist_ok=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=False)


## 2 — Clone da branch + prova de versão

In [ ]:
import os, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/vitorGgC569/reimagined.git'
REPO_ROOT = Path('/content/reimagined'); BRANCH = 'nsos-gpu-phases12'
TOKEN = None
try:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'GH_TOKEN'):
        try: TOKEN = userdata.get(key)
        except Exception: TOKEN = None
        if TOKEN: break
except Exception: pass
url = REPO_URL.replace('https://', f'https://oauth2:{TOKEN}@') if TOKEN else REPO_URL
if REPO_ROOT.exists():
    subprocess.run(['git','-C',str(REPO_ROOT),'remote','set-url','origin',url], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'fetch','--depth','1','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'reset','--hard',f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,url,str(REPO_ROOT)], check=True)
subprocess.run(['git','-C',str(REPO_ROOT),'remote','set-url','origin',REPO_URL], check=False)
os.environ['REPO_ROOT'] = str(REPO_ROOT)
sha = subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip()
mamba = (REPO_ROOT/'OXN/nsos/src/mamba2.cpp').read_text('utf-8')
assert 'proper_selective_ssm' in mamba and 'forward_proper_step' in mamba, 'fonte stale'
print(f'[ver] HEAD={sha} | Mamba proper + passo on-device: OK')


## 3 — Build `nsos_ext` + gate de gradcheck (CUDA T4 sm_75)

In [ ]:
import os, sys, subprocess, shutil, glob
from pathlib import Path
if 'nsos_ext' in sys.modules:
    raise RuntimeError('nsos_ext ja carregado — Runtime > Restart.')
PYTAG = f'cp{sys.version_info.major}{sys.version_info.minor}'
sha = subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip()
BUILD = str(REPO_ROOT / 'OXN/nsos/build-genbench')
EXT_DIR = Path('/content/nsos_ext_genbench'); EXT_DIR.mkdir(exist_ok=True)
CACHE_SO = DRIVE_ROOT / '_bootstrap' / f'nsos_ext_genbench.{PYTAG}.{sha}.so'
subprocess.run([sys.executable,'-m','pip','install','-q','pybind11','numpy'], check=True)
if CACHE_SO.exists():
    shutil.copy2(CACHE_SO, EXT_DIR/'nsos_ext.so'); os.environ['NSOS_BUILD_GENBENCH']=BUILD
    print('[build] .so do cache:', CACHE_SO)
else:
    r = subprocess.run(['cmake','-S',str(REPO_ROOT/'OXN/nsos'),'-B',BUILD,
                        '-DNSOS_ENABLE_CUDA=ON','-DCMAKE_CUDA_ARCHITECTURES=75',
                        '-DNSOS_BUILD_PYTHON=ON','-DNSOS_BUILD_TESTS=ON',
                        '-DNSOS_BUILD_CLI=OFF','-DNSOS_BUILD_API=OFF','-DNSOS_BUILD_OXTAMEM=OFF',
                        '-DCMAKE_BUILD_TYPE=Release',f'-DPython3_EXECUTABLE={sys.executable}'],
                       capture_output=True, text=True)
    print(r.stdout[-1000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-3000:]); raise RuntimeError('cmake configure falhou')
    r = subprocess.run(['cmake','--build',BUILD,'-j','2','--target','nsos_ext','test_gradcheck'],
                       capture_output=True, text=True)
    print(r.stdout[-1000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-3000:]); raise RuntimeError('build falhou')
    so = sorted(glob.glob(f'{BUILD}/**/nsos_ext*.so', recursive=True)); assert so
    shutil.copy2(so[0], EXT_DIR/'nsos_ext.so'); CACHE_SO.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(so[0], CACHE_SO); os.environ['NSOS_BUILD_GENBENCH']=BUILD
    print('[build] ok')
B = os.environ['NSOS_BUILD_GENBENCH']
for cand in (f'{B}/test_gradcheck', f'{B}/Release/test_gradcheck', f'{B}/test_gradcheck.exe'):
    if os.path.exists(cand):
        rr = subprocess.run([cand], capture_output=True, text=True)
        print('[gradcheck]', 'PASS' if rr.returncode==0 else 'FAIL'); break


## 4 — Tarefas + avaliação (em FLOAT e em TERNÁRIO)

**AR:** prompt `[BOS k1 v1 ... kN vN SEP q]`, resposta `[valor de q]` (loss = só o
valor → 100% sinal de recall). `n_kv` = capacidade. **Selective copy:** campo com K
dados entre BLANK + MARK → resposta = os K dados; treina curto, testa longo.
`eval_*` recebe `float_mode`: True = modo-treino (reference path, float);
False = modo-eval (forward ternário 1.58-bit).

In [ ]:
import random, numpy as np

def make_ar(rng, n_kv, n_key, n_val):
    # vocabularios DISJUNTOS: chaves em [0,n_key), valores em [n_key,n_key+n_val).
    # (id ja diz se e chave ou valor -> circuito de recall limpo; era a causa do AR
    # nao treinar na v3, onde chave/valor compartilhavam o vocab.)
    BOS, SEP = n_key + n_val, n_key + n_val + 1
    keys = rng.sample(range(n_key), n_kv)
    vals = [n_key + rng.randrange(n_val) for _ in range(n_kv)]
    kv = dict(zip(keys, vals)); q = rng.choice(keys)
    prompt = [BOS]
    for k, v in zip(keys, vals): prompt += [k, v]
    prompt += [SEP, q]
    return prompt, [kv[q]]

def make_selcopy(rng, field_len, n_data, n_sym):
    BLANK, MARK, BOS = n_sym, n_sym + 1, n_sym + 2
    pos = sorted(rng.sample(range(field_len), n_data)); data = [rng.randrange(n_sym) for _ in range(n_data)]
    field = [BLANK] * field_len
    for p, d in zip(pos, data): field[p] = d
    return [BOS] + field + [MARK], data

def eval_ar(model, data, V):
    model.set_training_mode(False)
    cor = tot = 0
    for prompt, ans in data:
        lg = np.asarray(model.forward_ids(prompt).cpu().numpy()).reshape(-1, V)
        cor += int(int(np.argmax(lg[-1])) == ans[0]); tot += 1
    return cor / max(tot, 1)

def eval_copy(model, data, V):
    model.set_training_mode(False)
    cor = tot = 0
    for prompt, ans in data:
        seq = prompt + ans
        lg = np.asarray(model.forward_ids(seq).cpu().numpy()).reshape(-1, V)
        ok = all(int(np.argmax(lg[len(prompt) + j - 1])) == ans[j] for j in range(len(ans)))
        cor += int(ok); tot += 1
    return cor / max(tot, 1)

# Nota: na GPU o forward e float (reference path) em treino E inferencia -> NAO ha
# gap ternario aqui (confirmado: float==ternario na v3). O 1.58-bit so morde no
# caminho empacotado CPU/dp4a, fora do escopo desta comparacao de mixer.
print('[tasks] AR (vocab disjunto) + selective-copy definidos')


## 5 — Variantes do MIXER + treino em batch

Atenção-pura / Mamba-puro (proper SSM) / híbrido NSOS, mesmo tamanho. MoE/KAN/TTT
OFF (compara o mixer). `train_supervised_batch` (batches homogêneos), loss impressa,
`first_token_loss_scale=eos_loss_scale=1.0`. (Na GPU o treino já é float; o ternário
só aparece no forward de inferência.)

In [ ]:
import os, sys, random
os.environ['NSOS_MAMBA_PROPER_SSM']  = '1'
os.environ['NSOS_MAMBA_CONV_K']      = '3'
os.environ['NSOS_MAMBA_A_LOGSPACED'] = '1'
os.environ['NSOS_GPU_POOL']          = '1'
sys.path.insert(0, str(__import__('pathlib').Path('/content/nsos_ext_genbench')))
import nsos_ext as nsos
dev = nsos.Device.GPU
NUM_LAYERS = int(os.environ.get('NSOS_GEN_LAYERS', '4'))
DMODEL = int(os.environ.get('NSOS_GEN_DMODEL', '128'))

def build_variant(variant, V):
    c = nsos.ModelConfig()
    c.num_layers = NUM_LAYERS; c.d_model = DMODEL; c.vocab_size = V
    c.n_heads = 4; c.n_kv_heads = 2; c.max_context_tokens = 1024
    c.use_exact_attention_training = True; c.use_flash_attn = False
    c.use_moe = False; c.use_kan = False; c.use_ttt = False; c.dropout = 0.0
    if variant == 'attn':
        c.attention_period = 1; c.attention_slot = 0
    elif variant == 'mamba':
        c.attention_period = 4; c.attention_slot = 4
    else:
        c.attention_period = 2; c.attention_slot = 1
    m = nsos.JambaModel(c, dev); m.to(dev)
    return m

def train_batched(model, lr, steps, batch, sampler, seed, label):
    tr = nsos.Trainer(model, lr); tr.warmup_steps = max(50, steps // 10)
    tr.total_training_steps = steps
    tr.first_token_loss_scale = 1.0; tr.eos_loss_scale = 1.0
    rng = random.Random(seed * 991 + 7)
    model.set_training_mode(True)
    win = max(1, steps // 6); run = 0.0
    for s in range(steps):
        bs = sampler(rng); ps, ans = [], []
        for _ in range(batch):
            p, a = bs(); ps.append(p); ans.append(a)
        run += tr.train_supervised_batch(ps, ans)
        if (s + 1) % win == 0:
            print(f'    [{label}] step {s+1}/{steps}  loss~{run/win:.3f}'); run = 0.0
    model.set_training_mode(False)

VARIANTS = ['attn', 'mamba', 'hybrid']
SEEDS = [int(x) for x in os.environ.get('NSOS_GEN_SEEDS', '0,1').split(',')]
STEPS = int(os.environ.get('NSOS_GEN_STEPS', '1500'))
BATCH = int(os.environ.get('NSOS_GEN_BATCH', '16'))
print(f'[cfg] L={NUM_LAYERS} d={DMODEL} | {VARIANTS} | seeds={SEEDS} | steps={STEPS} batch={BATCH} (={STEPS*BATCH} ex/run)')


## 5b — Sanidade: o trainer decora 1 exemplo?

Treina a atenção pra **decorar UM** exemplo de AR (vocab disjunto, n_kv=4). A loss
tem que ir a ~0 e o pred bater. Se não, o problema é trainer/gradiente (não dados
nem arquitetura). (Já confirmado nas runs anteriores; fica como gate rápido.)

In [ ]:
import numpy as np, random
_NK = 16; _NV = 16; _V = _NK + _NV + 2
nsos.set_seed(0)
mdbg = build_variant('attn', _V)
ex_p, ex_a = make_ar(random.Random(1), 4, _NK, _NV)   # 1 exemplo fixo, n_kv=4
tr = nsos.Trainer(mdbg, 3e-3); tr.total_training_steps = 400
tr.first_token_loss_scale = 1.0; tr.eos_loss_scale = 1.0
mdbg.set_training_mode(True)
for s in range(400):
    L = tr.train_supervised(ex_p, ex_a)
    if s % 100 == 0 or s == 399: print(f'  overfit step {s:3d}: loss={L:.4f}')
mdbg.set_training_mode(False)
lg = np.asarray(mdbg.forward_ids(ex_p).cpu().numpy()).reshape(-1, _V)
pred = int(np.argmax(lg[-1]))
print('=' * 60)
print(f'  alvo={ex_a[0]}  pred={pred}  -> {"OK (trainer decora 1 exemplo)" if pred==ex_a[0] else "FALHOU (trainer quebrado)"}')
print('=' * 60)


## 6 — Recall associativo: capacidade (vocab disjunto; treina n_kv=8; testa 4/8/16)

Baseline = 1/n_val. **Gate:** atenção tem que acertar bem em n_kv=8 (controle
positivo). Mais steps + LR maior que o copy (recall é mais difícil de induzir).

In [ ]:
import time, numpy as np, random
N_KEY = int(os.environ.get('NSOS_AR_KEYS', '16')); N_VAL = int(os.environ.get('NSOS_AR_VALS', '16'))
V = N_KEY + N_VAL + 2; base = 1.0 / N_VAL; KV_TEST = [4, 8, 16]
AR_STEPS = int(os.environ.get('NSOS_AR_STEPS', str(STEPS * 3)))   # recall e mais dificil que copy
AR_LR = float(os.environ.get('NSOS_AR_LR', '3e-3'))
ar = {}; t0 = time.time()
for variant in VARIANTS:
    acc = {k: [] for k in KV_TEST}
    for seed in SEEDS:
        nsos.set_seed(seed); m = build_variant(variant, V)
        train_batched(m, AR_LR, AR_STEPS, BATCH, lambda r: (lambda: make_ar(r, 8, N_KEY, N_VAL)), seed, f'AR {variant} s{seed}')
        ev = random.Random(seed * 13 + 5)
        for k in KV_TEST:
            d = [make_ar(ev, k, N_KEY, N_VAL) for _ in range(300)]
            acc[k].append(eval_ar(m, d, V))
        print(f'  [{variant:6s} s{seed}] ' + ' '.join(f'n_kv={k}:{acc[k][-1]:.2f}' for k in KV_TEST))
    ar[variant] = {k: float(np.mean(acc[k])) for k in KV_TEST}
print('\n' + '=' * 72)
print(f'RECALL ASSOCIATIVO (baseline={base:.3f}; treina n_kv=8, {AR_STEPS} steps)   ({time.time()-t0:.0f}s)')
print(f'{"variante":<10}' + ''.join(f'{("n_kv="+str(k)):>14}' for k in KV_TEST))
for v in VARIANTS:
    print(f'  {v:<8}' + ''.join(f'{ar[v][k]:>14.3f}' for k in KV_TEST))
print('=' * 72)


## 7 — Selective copy: extrapolação de comprimento (treina ≤64; testa 64/128/256)

Já funcionou nas runs anteriores (harness validado). Mede quem **extrapola** melhor
pra campos mais longos que o treino — o upside estrutural do SSM.

In [ ]:
import time, numpy as np, random
N_SYM_C = int(os.environ.get('NSOS_COPY_SYM', '20')); N_DATA = 4
LT = int(os.environ.get('NSOS_COPY_LEN', '64')); Vc = N_SYM_C + 3
base_c = (1.0 / N_SYM_C) ** N_DATA; TEST_LENS = [LT, 2 * LT, 4 * LT]
copy = {}; t0 = time.time()
for variant in VARIANTS:
    acc = {L: [] for L in TEST_LENS}
    for seed in SEEDS:
        nsos.set_seed(seed); m = build_variant(variant, Vc)
        def sampler(r):
            L = r.randint(16, LT)
            return lambda: make_selcopy(r, L, N_DATA, N_SYM_C)
        train_batched(m, 2e-3, STEPS, BATCH, sampler, seed, f'COPY {variant} s{seed}')
        ev = random.Random(seed * 29 + 3)
        for L in TEST_LENS:
            d = [make_selcopy(ev, L, N_DATA, N_SYM_C) for _ in range(300)]
            acc[L].append(eval_copy(m, d, Vc))
        print(f'  [{variant:6s} s{seed}] ' + ' '.join(f'L={L}:{acc[L][-1]:.2f}' for L in TEST_LENS))
    copy[variant] = {L: float(np.mean(acc[L])) for L in TEST_LENS}
print('\n' + '=' * 72)
print(f'SELECTIVE COPY exact-match (baseline~{base_c:.0e}; treino campo<=64)   ({time.time()-t0:.0f}s)')
print(f'{"variante":<10}' + ''.join(f'{("L="+str(L)):>14}' for L in TEST_LENS))
for v in VARIANTS:
    print(f'  {v:<8}' + ''.join(f'{copy[v][L]:>14.3f}' for L in TEST_LENS))
print('=' * 72)


## 8 — VEREDITO (gate = atenção em FLOAT)

In [ ]:
def g(c): return 'OK' if c else 'FALHOU'
a8 = ar['attn'][8]; GATE = a8 >= 0.80
print('=' * 72)
print(f'GATE — atencao resolve AR n_kv=8?  attn={a8:.3f}  -> {g(GATE)}')
if not GATE:
    print('  >> AR ainda nao treinou no controle de atencao. Suba NSOS_AR_STEPS')
    print('     (ex. 9000) e/ou NSOS_GEN_BATCH (32). (o copy abaixo ja valida o')
    print('     harness; recall e mais dificil de induzir que copy.)')
else:
    a16, h16, m16 = ar['attn'][16], ar['hybrid'][16], ar['mamba'][16]
    print('\n1) TABLE STAKES (AR n_kv=8):')
    for v in VARIANTS: print(f'   {v:<8} {ar[v][8]:.3f} -> {g(ar[v][8] > 0.7)}')
    print('\n2) DISCRIMINADOR — hibrido ~ atencao na CAPACIDADE (n_kv=16)?')
    print(f'   attn={a16:.3f} hibrido={h16:.3f} mamba={m16:.3f} | hibrido>=0.85*attn: {g(h16>=0.85*a16)}')
print('\n3) UPSIDE — extrapolacao de comprimento (selective copy):')
for v in VARIANTS:
    print(f'   {v:<8} treino(L={LT})={copy[v][LT]:.3f}  L={2*LT}={copy[v][2*LT]:.3f}  L={4*LT}={copy[v][4*LT]:.3f}')
best = max(VARIANTS, key=lambda v: copy[v][2 * LT])
print(f'   melhor extrapolador (L={2*LT}): {best} = {copy[best][2*LT]:.3f}')
print('\nLEITURA: AR julga memoria associativa (recall); copy julga extrapolacao de')
print('comprimento. Vale escalar se attn passa o gate, hibrido acompanha no recall,')
print('e Mamba/hibrido extrapolam melhor que a atencao.')
print('=' * 72)
